# KESARi Inbound — Translation Pipeline

This notebook runs the full pipeline:
1. **Scrape** — downloads all pages from the live KESARi website into `INT/ENG/`
2. **Detect changes** — compares SHA-256 hashes with previous run
3. **Translate** — sends changed pages to Gemini API, saves to `INT/{lang}/`

**Run this notebook once to do a full translation. After that, run it daily to catch changes.**

## Step 0 — Setup
Install dependencies (run once)

In [ ]:
!pip install requests beautifulsoup4 google-generativeai python-dotenv lxml

## Step 1 — Configure API Key
Get your free Gemini key from https://aistudio.google.com → Get API Key
Then paste it below OR add it to `pipeline/.env` as `GEMINI_API_KEY=your_key`

In [ ]:
import os

# Option A — paste key directly (for testing)
# os.environ['GEMINI_API_KEY'] = 'paste-your-key-here'

# Option B — load from .env file (recommended)
from dotenv import load_dotenv
load_dotenv('.env')

if not os.environ.get('GEMINI_API_KEY'):
    print('⚠️  GEMINI_API_KEY not set! Add it to pipeline/.env or paste above.')
else:
    print('✔ API key loaded')

## Step 2 — Check Configuration

In [ ]:
from config import BASE_URL, ENG_DIR, INT_DIR, LANGUAGES, CHECK_INTERVAL_HOURS

print(f'Source URL     : {BASE_URL}')
print(f'English folder : {ENG_DIR}')
print(f'Output folder  : {INT_DIR}')
print(f'Languages      : {list(LANGUAGES.keys())}')
print(f'Check every    : {CHECK_INTERVAL_HOURS} hours')

## Step 3 — Scrape the Website
Downloads all pages from the live KESARi website. Only saves pages that have changed.

In [ ]:
from scraper import scrape

print(f'Scraping {BASE_URL}...\n')
changed_pages = scrape(verbose=True)

print(f'\n── Result ──')
print(f'{len(changed_pages)} page(s) changed')
for p in changed_pages:
    print(f'  {p}')

## Step 4 — Translate Changed Pages
Only translates pages that changed in Step 3. If nothing changed, this is a no-op.

In [ ]:
from translator import translate_changed_pages

if not changed_pages:
    print('No pages changed — nothing to translate.')
else:
    print(f'Translating {len(changed_pages)} changed page(s) into {len(LANGUAGES)} languages...\n')
    translate_changed_pages(changed_pages)
    print('\n✔ Done!')

## Step 5 (Optional) — Force Translate ALL Pages
Run this only if you want to re-translate everything from scratch (e.g. first run or language added).

In [ ]:
# Uncomment to run full translation
# from translator import translate_all_pages
# translate_all_pages()

## Step 6 — Check Output Structure

In [ ]:
import os, glob

for lang in ['ENG'] + list(LANGUAGES.keys()):
    folder = os.path.join(INT_DIR, lang)
    files  = glob.glob(os.path.join(folder, '**', '*.html'), recursive=True)
    print(f'  INT/{lang}/  → {len(files)} HTML files')

## Step 7 — Schedule Daily Runs
To run this pipeline automatically every 24 hours, use the scheduler cell below.

In [ ]:
import time, datetime
from scraper import scrape
from translator import translate_changed_pages
from config import CHECK_INTERVAL_HOURS

def run_pipeline():
    print(f'\n[{datetime.datetime.now():%Y-%m-%d %H:%M}] Running pipeline...')
    changed = scrape(verbose=False)
    if changed:
        print(f'  {len(changed)} page(s) changed — translating...')
        translate_changed_pages(changed)
        print('  ✔ Done')
    else:
        print('  No changes detected')

# Uncomment below to run scheduler (keeps notebook running)
# while True:
#     run_pipeline()
#     print(f'  Next check in {CHECK_INTERVAL_HOURS}h...')
#     time.sleep(CHECK_INTERVAL_HOURS * 3600)